# Session 10 - AprilTag
- info
- pick up when correct ID
- centralise

In [ ]:
pip install opencv-python numpy

In [ ]:
import numpy as np
import cv2
import time
from IPython.display import clear_output

from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.88.1")

got.load_models(["apriltag_qrcode"])
got.open_camera()

In [ ]:
while True:
    try:
        tags = got.get_apriltag_total_info()
        clear_output()
        # [ [id, center_x, cy, height, width, area, distance5, distance7, distance10, 
        #    x, y, z, bearingAngle_h, bearingAngle_v], … ]
        print(tags)
        time.sleep(0.5)
    except KeyboardInterrupt:
        break

In [ ]:
# print ID
while True:
    frame = got.read_camera_data()
    if frame is not None:
        nparr = np.frombuffer(frame, np.uint8)
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    tags = got.get_apriltag_total_info()
    # [ [id, cx, center_y, height, width, area, distance5, distance7, distance10, 
    #    x, y, z, bearingAngle_h, bearingAngle_v], … ]
    if tags:
        tag = tags[0]
        cv2.putText(data, f"ID: {tag[0]}", (20, 20), cv2.FONT_HERSHEY_SIMPLEX, 
                    1, (0, 255, 0), 3, cv2.LINE_AA)

    cv2.imshow("UGOT camera", data)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [ ]:
# bounding box
while True:
    frame = got.read_camera_data()
    if frame is not None:
        nparr = np.frombuffer(frame, np.uint8)
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    tags = got.get_apriltag_total_info()
    # [ [id, center_x, center_y, height, width, area, distance5, distance7, distance10, 
    #    x, y, z, bearingAngle_h, bearingAngle_v], … ]
    if tags:
        tag = tags[0] # get the first tag

        cv2.putText(data, f"ID: {tag[0]}", (20, 20), cv2.FONT_HERSHEY_SIMPLEX, 
                    1, (0, 255, 0), 3, cv2.LINE_AA)
        
        cx = tag[1]
        cy = tag[2]
        height = tag[3]
        width = tag[4]
        # draw bounding box
        cv2.rectangle(data, 
                        (int(cx-width//2), int(cy-height//2)), 
                        (int(cx+width//2), int(cy+height//2)), 
                        (255, 0, 0), 2)

    cv2.imshow("UGOT camera", data)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

In [ ]:
# centralise
got.balance_start_balancing()
while True:
    frame = got.read_camera_data()
    if frame is not None:
        nparr = np.frombuffer(frame, np.uint8)
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    tags = got.get_apriltag_total_info()
    # [ [id, center_x, center_y, height, width, area, distance5, distance7, distance10, 
    #    x, y, z, bearingAngle_h, bearingAngle_v], … ]
    if tags:
        tag = tags[0] # get the first tag

        cv2.putText(data, f"ID: {tag[0]}", (20, 20), cv2.FONT_HERSHEY_SIMPLEX, 
                    1, (0, 255, 0), 3, cv2.LINE_AA)
        
        cx = tag[1]
        cy = tag[2]
        height = tag[3]
        width = tag[4]
        # draw bounding box
        cv2.rectangle(data,
                        (int(cx-width//2), int(cy-height//2)), 
                        (int(cx+width//2), int(cy+height//2)), 
                        (255, 0, 0), 2)
        
        # centralise apriltag
        if cx > 330:
            got.balance_turn_speed(3, 5)
        elif cx < 310:
            got.balance_turn_speed(2, 5)
        else:
            got.mecanum_stop()
    # if apriltag not found, keep turning faster
    else:
        got.balance_turn_speed(3, 10)

    cv2.imshow("UGOT camera", data)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()
got.mecanum_stop()